# 🚀 Improved Time Series Forecasting - Ensemble Methods Research

Notebook này nghiên cứu và cải thiện phương pháp ensemble để giảm MSE tốt hơn so với model đơn lẻ tốt nhất (NLinear).

## Mục tiêu:
1. **Tối ưu NLinear model** - Model tốt nhất hiện tại (MSE: 446.75)
2. **Nghiên cứu phương pháp Ensemble mới**:
   - Stacking với meta-learner
   - Selective Ensemble (chọn top-k models)
   - Dynamic Weighting (weight theo time step)
   - Correlation-based Ensemble
   - Residual-based Weighting
3. **So sánh và chọn phương pháp tốt nhất**

## Kết quả hiện tại:
- Best Single Model: **NLinear** (MSE: 446.75)
- Ensemble hiện tại: **MSE: 787.41** (tệ hơn 76.25%)


## 1. Setup và Import Libraries


In [1]:
# Cài đặt các thư viện cần thiết
import subprocess
import sys

def install_package(package, import_name=None):
    """Cài đặt package nếu chưa có"""
    if import_name is None:
        import_name = package
    try:
        __import__(import_name)
        print(f"✓ {package} đã được cài đặt")
        return True
    except ImportError:
        print(f"📦 Đang cài đặt {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package], 
                                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"✓ Đã cài đặt {package}")
            return True
        except Exception as e:
            print(f"⚠️  Lỗi khi cài đặt {package}: {e}")
            return False

# Cài đặt các thư viện cần thiết
packages_to_install = [
    ('neuralforecast', 'neuralforecast'),
    ('optuna', 'optuna'),
    ('scikit-learn', 'sklearn'),
    ('scipy', 'scipy')
]

print("🔧 Kiểm tra và cài đặt các thư viện cần thiết...\n")
for package, import_name in packages_to_install:
    install_package(package, import_name)

print("\n✓ Hoàn thành kiểm tra/cài đặt thư viện!")


🔧 Kiểm tra và cài đặt các thư viện cần thiết...

📦 Đang cài đặt neuralforecast...
✓ Đã cài đặt neuralforecast
✓ optuna đã được cài đặt
✓ scikit-learn đã được cài đặt
✓ scipy đã được cài đặt

✓ Hoàn thành kiểm tra/cài đặt thư viện!


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import pearsonr
import os
from pathlib import Path

# Set encoding để tránh lỗi Unicode
import sys
import io
if sys.platform == 'win32':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8')

# NeuralForecast
from neuralforecast import NeuralForecast
from neuralforecast.models import NLinear, DLinear, NBEATS, NHITS, PatchTST, TimesNet
import optuna
from optuna import Trial

print("✓ Đã import các thư viện cần thiết")


✓ Đã import các thư viện cần thiết


## 2. Load và Chuẩn Bị Dữ Liệu


In [3]:
# Load dữ liệu
csv_path = Path("./FPT_train.csv")
if not csv_path.exists():
    # Thử download từ Google Drive
    from pathlib import Path
    import subprocess
    DRIVE_FILE_ID = "1nS9xshut38SJEX__PD_zjKFtj2CQCn7S"
    try:
        import gdown
        gdown.download(f"https://drive.google.com/uc?id={DRIVE_FILE_ID}", str(csv_path), quiet=False)
    except:
        print("⚠️  Vui lòng đảm bảo file FPT_train.csv tồn tại")
        raise

df = pd.read_csv(csv_path, parse_dates=["time"])
df = df.sort_values("time").reset_index(drop=True)

print(f"✓ Đã load dữ liệu: {len(df)} điểm")
print(f"   - Từ {df['time'].min()} đến {df['time'].max()}")

# Chuẩn bị dữ liệu
target_col = "close"
horizon = 100  # Dự đoán 100 ngày tiếp theo

close_values = df[target_col].values.astype("float32")
T = len(close_values)

# Chia train/validation/test
train_size = int(T * 0.8)
val_size = int(T * 0.1)

train_data = close_values[:train_size]
val_data = close_values[train_size:train_size + val_size]
test_data = close_values[train_size + val_size:]

print(f"\n📊 Chia dữ liệu:")
print(f"   - Train: {len(train_data)} điểm")
print(f"   - Val: {len(val_data)} điểm")
print(f"   - Test: {len(test_data)} điểm")


Downloading...
From: https://drive.google.com/uc?id=1nS9xshut38SJEX__PD_zjKFtj2CQCn7S
To: /content/FPT_train.csv
100%|██████████| 55.3k/55.3k [00:00<00:00, 76.8MB/s]

✓ Đã load dữ liệu: 1149 điểm
   - Từ 2020-08-03 00:00:00 đến 2025-03-10 00:00:00

📊 Chia dữ liệu:
   - Train: 919 điểm
   - Val: 114 điểm
   - Test: 116 điểm


In [4]:
# Chuẩn bị dữ liệu cho NeuralForecast
train_nf = pd.DataFrame({
    'unique_id': 'FPT',
    'ds': pd.date_range(start=df['time'].iloc[0], periods=len(train_data), freq='D'),
    'y': train_data
})

val_nf = pd.DataFrame({
    'unique_id': 'FPT',
    'ds': pd.date_range(start=df['time'].iloc[train_size], periods=len(val_data), freq='D'),
    'y': val_data
})

# Full train (train + val) để train final model
train_nf_full = pd.concat([train_nf, val_nf], ignore_index=True)

# Test data (ground truth cho 100 ngày cuối)
test_nf = pd.DataFrame({
    'unique_id': 'FPT',
    'ds': pd.date_range(start=df['time'].iloc[train_size + val_size], periods=len(test_data), freq='D'),
    'y': test_data
})

# Ground truth cho 100 ngày cuối (nếu có)
y_true = test_data[:horizon] if len(test_data) >= horizon else test_data

print(f"✓ Đã chuẩn bị dữ liệu cho NeuralForecast")
print(f"   - Train: {len(train_nf)} điểm")
print(f"   - Val: {len(val_nf)} điểm")
print(f"   - Test ground truth: {len(y_true)} điểm")


✓ Đã chuẩn bị dữ liệu cho NeuralForecast
   - Train: 919 điểm
   - Val: 114 điểm
   - Test ground truth: 100 điểm


## 3. Train Tất Cả Models Để Có Predictions Cho Ensemble


In [ ]:
# Train tất cả models với hyperparameters đã tối ưu từ notebook trước
print("="*70)
print("🔧 TRAINING TẤT CẢ MODELS")
print("="*70)

# Function để clear GPU memory
def clear_gpu_memory():
    """Clear GPU memory cache"""
    try:
        import torch
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print("   ✓ Đã clear GPU memory")
    except:
        pass

all_predictions = {}
all_results = []

# Best parameters từ notebook trước (điều chỉnh để giảm memory usage)
# Giảm num_kernels cho TimesNet để tránh OutOfMemoryError
best_params = {
    'NLinear': {'input_size': 200, 'learning_rate': 0.001, 'max_steps': 200},
    'DLinear': {'input_size': 200, 'learning_rate': 0.001, 'max_steps': 200},
    'NBEATS': {'input_size': 200, 'learning_rate': 0.001, 'max_steps': 200},
    'NHITS': {'input_size': 200, 'learning_rate': 0.001, 'max_steps': 200},
    'PatchTST': {'input_size': 200, 'patch_len': 16, 'stride': 8, 'learning_rate': 0.001, 'max_steps': 200},
    'TimesNet': {'input_size': 200, 'top_k': 5, 'num_kernels': 32, 'learning_rate': 0.001, 'max_steps': 200}  # Giảm từ 64 xuống 32
}

# Helper function để train và evaluate một model
def train_and_evaluate_model(model_name, model_class, model_params, train_df, y_true_val):
    """Train một model và return predictions + metrics"""
    try:
        print(f"\n🔄 Training {model_name}...")
        
        # Tạo model
        if model_name == 'TimesNet':
            model = model_class(
                h=horizon,
                input_size=model_params['input_size'],
                top_k=model_params['top_k'],
                num_kernels=model_params['num_kernels'],
                learning_rate=model_params['learning_rate'],
                max_steps=model_params['max_steps'],
                val_check_steps=10,
                batch_size=32,  # Giảm batch_size để tiết kiệm memory
            )
        elif model_name == 'PatchTST':
            model = model_class(
                h=horizon,
                input_size=model_params['input_size'],
                patch_len=model_params['patch_len'],
                stride=model_params['stride'],
                learning_rate=model_params['learning_rate'],
                max_steps=model_params['max_steps'],
                val_check_steps=10,
                batch_size=32,  # Giảm batch_size
            )
        else:
            model = model_class(
                h=horizon,
                input_size=model_params['input_size'],
                learning_rate=model_params['learning_rate'],
                max_steps=model_params['max_steps'],
                val_check_steps=10,
                batch_size=64,  # Batch size cho các models nhẹ hơn
            )
        
        # Train
        nf_model = NeuralForecast(models=[model], freq='D')
        nf_model.fit(df=train_df, val_size=0)
        
        # Predict
        forecast = nf_model.predict()
        pred_col = [col for col in forecast.columns if col not in ['unique_id', 'ds']][0]
        pred = forecast[pred_col].values[:len(y_true_val)]
        
        # Evaluate
        mse = mean_squared_error(y_true_val, pred)
        mae = mean_absolute_error(y_true_val, pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true_val, pred)
        mape = np.mean(np.abs((y_true_val - pred) / y_true_val)) * 100
        
        # Clear memory
        del model, nf_model, forecast
        clear_gpu_memory()
        
        return {
            'success': True,
            'prediction': pred,
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'R²': r2,
            'MAPE': mape
        }
    except Exception as e:
        print(f"   ⚠️  Lỗi khi train {model_name}: {e}")
        clear_gpu_memory()
        return {'success': False, 'error': str(e)}

# Train từng model một và clear memory sau mỗi model
models_to_train = [
    ('NLinear', NLinear, best_params['NLinear']),
    ('DLinear', DLinear, best_params['DLinear']),
    ('NBEATS', NBEATS, best_params['NBEATS']),
    ('NHITS', NHITS, best_params['NHITS']),
    ('PatchTST', PatchTST, best_params['PatchTST']),
    ('TimesNet', TimesNet, best_params['TimesNet']),
]

# Clear memory trước khi bắt đầu
clear_gpu_memory()

for i, (model_name, model_class, model_params) in enumerate(models_to_train, 1):
    print(f"\n{'='*70}")
    print(f"{i}️⃣  Training {model_name}...")
    print("="*70)
    
    result = train_and_evaluate_model(model_name, model_class, model_params, train_nf_full, y_true)
    
    if result['success']:
        all_predictions[model_name] = result['prediction']
        all_results.append({
            'Model': model_name,
            'MSE': result['MSE'],
            'RMSE': result['RMSE'],
            'MAE': result['MAE'],
            'R²': result['R²'],
            'MAPE': result['MAPE']
        })
        print(f"   ✓ {model_name} - MSE: {result['MSE']:.4f}, RMSE: {result['RMSE']:.4f}")
    else:
        print(f"   ❌ {model_name} - Failed: {result.get('error', 'Unknown error')}")
        # Nếu TimesNet fail, có thể skip hoặc thử với num_kernels nhỏ hơn
        if model_name == 'TimesNet':
            print("   💡 Thử TimesNet với num_kernels=16...")
            model_params_small = model_params.copy()
            model_params_small['num_kernels'] = 16
            result2 = train_and_evaluate_model(model_name, model_class, model_params_small, train_nf_full, y_true)
            if result2['success']:
                all_predictions[model_name] = result2['prediction']
                all_results.append({
                    'Model': model_name,
                    'MSE': result2['MSE'],
                    'RMSE': result2['RMSE'],
                    'MAE': result2['MAE'],
                    'R²': result2['R²'],
                    'MAPE': result2['MAPE']
                })
                print(f"   ✓ {model_name} (num_kernels=16) - MSE: {result2['MSE']:.4f}, RMSE: {result2['RMSE']:.4f}")
    
    # Clear memory sau mỗi model
    clear_gpu_memory()

print("\n" + "="*70)
print("✓ Đã train tất cả models!")
print(f"   - Số models thành công: {len(all_predictions)}/{len(models_to_train)}")
print("="*70)


🔧 TRAINING TẤT CẢ MODELS


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


   ✓ Đã clear GPU memory

1️⃣  Training NLinear...

🔄 Training NLinear...


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ linear       │ Linear        │ 20.1 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 20.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 20.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 4                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

   ✓ Đã clear GPU memory
   ✓ PatchTST - MSE: 68.1400, RMSE: 8.2547
   ✓ Đã clear GPU memory

6️⃣  Training TimesNet...

🔄 Training TimesNet...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss           │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train   │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler         │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ model          │ ModuleList    │  715 M │ train │     0 │
│ 4 │ enc_embedding  │ DataEmbedding │    192 │ train │     0 │
│ 5 │ layer_norm     │ LayerNorm     │    128 │ train │     0 │
│ 6 │ predict_linear │ Linear        │ 60.3 K │ train │     0 │
│ 7 │ projection     │ Linear        │     65 │ train │     0 │
└───┴────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 715 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 715 M                                                                                                
Total estimated model params size (MB): 2.9 K                                                                      
Modules in train mode: 154                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

## 4. Phương Pháp Ensemble Mới

### 4.1. Selective Ensemble - Chọn Top-K Models Tốt Nhất


In [ ]:
print("="*70)
print("🔧 SELECTIVE ENSEMBLE - Chọn Top-K Models Tốt Nhất")
print("="*70)

# Sắp xếp models theo MSE
sorted_results = sorted(all_results, key=lambda x: x['MSE'])
print("\n📊 Models sắp xếp theo MSE:")
for i, result in enumerate(sorted_results, 1):
    print(f"   {i}. {result['Model']}: MSE = {result['MSE']:.4f}")

# Thử các giá trị k khác nhau
selective_ensemble_results = []
for k in [1, 2, 3, 4, 5, 6]:
    # Chọn top-k models
    top_k_models = [r['Model'] for r in sorted_results[:k]]
    
    # Tính weights dựa trên inverse MSE
    weights = {}
    total_inv_mse = 0
    
    for model_name in top_k_models:
        mse = next(r['MSE'] for r in all_results if r['Model'] == model_name)
        inv_mse = 1.0 / (mse + 1e-10)
        weights[model_name] = inv_mse
        total_inv_mse += inv_mse
    
    # Normalize weights
    for model_name in weights:
        weights[model_name] = weights[model_name] / total_inv_mse
    
    # Tạo ensemble prediction
    ensemble_pred = np.zeros(len(y_true))
    for model_name in top_k_models:
        ensemble_pred += weights[model_name] * all_predictions[model_name]
    
    # Đánh giá
    mse_ens = mean_squared_error(y_true, ensemble_pred)
    mae_ens = mean_absolute_error(y_true, ensemble_pred)
    rmse_ens = np.sqrt(mse_ens)
    r2_ens = r2_score(y_true, ensemble_pred)
    mape_ens = np.mean(np.abs((y_true - ensemble_pred) / y_true)) * 100
    
    selective_ensemble_results.append({
        'k': k,
        'models': top_k_models,
        'MSE': mse_ens,
        'RMSE': rmse_ens,
        'MAE': mae_ens,
        'R²': r2_ens,
        'MAPE': mape_ens,
        'prediction': ensemble_pred.copy()
    })
    
    best_single_mse = sorted_results[0]['MSE']
    improvement = ((best_single_mse - mse_ens) / best_single_mse) * 100
    
    print(f"\n📊 Top-{k} Ensemble:")
    print(f"   - Models: {', '.join(top_k_models)}")
    print(f"   - MSE: {mse_ens:.4f}")
    print(f"   - RMSE: {rmse_ens:.4f}")
    print(f"   - Cải thiện so với best single: {improvement:+.2f}%")

# Tìm best selective ensemble
best_selective = min(selective_ensemble_results, key=lambda x: x['MSE'])
print(f"\n🏆 Best Selective Ensemble: Top-{best_selective['k']}")
print(f"   - MSE: {best_selective['MSE']:.4f}")
print(f"   - Models: {', '.join(best_selective['models'])}")


### 4.2. Stacking với Meta-Learner


In [ ]:
print("="*70)
print("🔧 STACKING VỚI META-LEARNER")
print("="*70)

# Chuẩn bị features cho meta-learner (predictions từ base models)
X_meta = np.column_stack([all_predictions[name] for name in sorted([k for k in all_predictions.keys()])])
y_meta = y_true

# Thử các meta-learners khác nhau
meta_learners = {
    'LinearRegression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Ridge (alpha=10.0)': Ridge(alpha=10.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1),
    'Lasso (alpha=1.0)': Lasso(alpha=1.0),
    'RandomForest (n=10)': RandomForestRegressor(n_estimators=10, random_state=42),
    'RandomForest (n=50)': RandomForestRegressor(n_estimators=50, random_state=42),
}

stacking_results = []

for name, meta_learner in meta_learners.items():
    try:
        # Train meta-learner
        meta_learner.fit(X_meta, y_meta)
        
        # Predict
        pred_stacking = meta_learner.predict(X_meta)
        
        # Đánh giá
        mse_st = mean_squared_error(y_meta, pred_stacking)
        mae_st = mean_absolute_error(y_meta, pred_stacking)
        rmse_st = np.sqrt(mse_st)
        r2_st = r2_score(y_meta, pred_stacking)
        mape_st = np.mean(np.abs((y_meta - pred_stacking) / y_meta)) * 100
        
        # Lấy coefficients/feature importance nếu có
        if hasattr(meta_learner, 'coef_'):
            coefs = meta_learner.coef_
        elif hasattr(meta_learner, 'feature_importances_'):
            coefs = meta_learner.feature_importances_
        else:
            coefs = None
        
        stacking_results.append({
            'meta_learner': name,
            'MSE': mse_st,
            'RMSE': rmse_st,
            'MAE': mae_st,
            'R²': r2_st,
            'MAPE': mape_st,
            'coefficients': coefs,
            'prediction': pred_stacking.copy()
        })
        
        best_single_mse = sorted_results[0]['MSE']
        improvement = ((best_single_mse - mse_st) / best_single_mse) * 100
        
        print(f"\n📊 Stacking với {name}:")
        print(f"   - MSE: {mse_st:.4f}")
        print(f"   - RMSE: {rmse_st:.4f}")
        print(f"   - Cải thiện so với best single: {improvement:+.2f}%")
        if coefs is not None:
            model_names = sorted([k for k in all_predictions.keys()])
            print(f"   - Top 3 weights: {dict(sorted(zip(model_names, coefs), key=lambda x: abs(x[1]), reverse=True)[:3])}")
    except Exception as e:
        print(f"   ⚠️  Lỗi với {name}: {e}")

# Tìm best stacking
if stacking_results:
    best_stacking = min(stacking_results, key=lambda x: x['MSE'])
    print(f"\n🏆 Best Stacking: {best_stacking['meta_learner']}")
    print(f"   - MSE: {best_stacking['MSE']:.4f}")


### 4.3. Correlation-based Ensemble - Loại Bỏ Models Có Correlation Cao


In [ ]:
print("="*70)
print("🔧 CORRELATION-BASED ENSEMBLE")
print("="*70)

# Tính correlation matrix giữa các predictions
model_names = sorted([k for k in all_predictions.keys()])
correlation_matrix = np.zeros((len(model_names), len(model_names)))

for i, name1 in enumerate(model_names):
    for j, name2 in enumerate(model_names):
        if i == j:
            correlation_matrix[i, j] = 1.0
        else:
            corr, _ = pearsonr(all_predictions[name1], all_predictions[name2])
            correlation_matrix[i, j] = corr

print("\n📊 Correlation Matrix giữa các models:")
corr_df = pd.DataFrame(correlation_matrix, index=model_names, columns=model_names)
print(corr_df.round(3))

# Chọn models có correlation thấp với nhau
# Strategy: Chọn models sao cho average correlation với nhau < threshold
thresholds = [0.7, 0.8, 0.9, 0.95]
correlation_ensemble_results = []

for threshold in thresholds:
    selected_models = []
    
    # Bắt đầu với model tốt nhất
    selected_models.append(sorted_results[0]['Model'])
    
    # Thêm các models khác nếu correlation với tất cả models đã chọn < threshold
    for result in sorted_results[1:]:
        model_name = result['Model']
        model_idx = model_names.index(model_name)
        
        # Kiểm tra correlation với tất cả models đã chọn
        max_corr = 0
        for selected in selected_models:
            selected_idx = model_names.index(selected)
            max_corr = max(max_corr, abs(correlation_matrix[model_idx, selected_idx]))
        
        if max_corr < threshold:
            selected_models.append(model_name)
    
    if len(selected_models) > 0:
        # Tính weights
        weights = {}
        total_inv_mse = 0
        
        for model_name in selected_models:
            mse = next(r['MSE'] for r in all_results if r['Model'] == model_name)
            inv_mse = 1.0 / (mse + 1e-10)
            weights[model_name] = inv_mse
            total_inv_mse += inv_mse
        
        # Normalize weights
        for model_name in weights:
            weights[model_name] = weights[model_name] / total_inv_mse
        
        # Tạo ensemble prediction
        ensemble_pred = np.zeros(len(y_true))
        for model_name in selected_models:
            ensemble_pred += weights[model_name] * all_predictions[model_name]
        
        # Đánh giá
        mse_ens = mean_squared_error(y_true, ensemble_pred)
        mae_ens = mean_absolute_error(y_true, ensemble_pred)
        rmse_ens = np.sqrt(mse_ens)
        r2_ens = r2_score(y_true, ensemble_pred)
        mape_ens = np.mean(np.abs((y_true - ensemble_pred) / y_true)) * 100
        
        # Tính average correlation giữa các models đã chọn
        avg_corr = 0
        count = 0
        for i, m1 in enumerate(selected_models):
            for j, m2 in enumerate(selected_models):
                if i < j:
                    idx1 = model_names.index(m1)
                    idx2 = model_names.index(m2)
                    avg_corr += abs(correlation_matrix[idx1, idx2])
                    count += 1
        avg_corr = avg_corr / count if count > 0 else 0
        
        correlation_ensemble_results.append({
            'threshold': threshold,
            'models': selected_models,
            'avg_correlation': avg_corr,
            'MSE': mse_ens,
            'RMSE': rmse_ens,
            'MAE': mae_ens,
            'R²': r2_ens,
            'MAPE': mape_ens,
            'prediction': ensemble_pred.copy()
        })
        
        best_single_mse = sorted_results[0]['MSE']
        improvement = ((best_single_mse - mse_ens) / best_single_mse) * 100
        
        print(f"\n📊 Correlation-based Ensemble (threshold={threshold}):")
        print(f"   - Selected models ({len(selected_models)}): {', '.join(selected_models)}")
        print(f"   - Average correlation: {avg_corr:.3f}")
        print(f"   - MSE: {mse_ens:.4f}")
        print(f"   - RMSE: {rmse_ens:.4f}")
        print(f"   - Cải thiện so với best single: {improvement:+.2f}%")

# Tìm best correlation-based ensemble
if correlation_ensemble_results:
    best_corr = min(correlation_ensemble_results, key=lambda x: x['MSE'])
    print(f"\n🏆 Best Correlation-based Ensemble (threshold={best_corr['threshold']}):")
    print(f"   - MSE: {best_corr['MSE']:.4f}")
    print(f"   - Models: {', '.join(best_corr['models'])}")


### 4.4. Dynamic Weighting - Weight Theo Từng Time Step


In [ ]:
print("="*70)
print("🔧 DYNAMIC WEIGHTING - Weight Theo Từng Time Step")
print("="*70)

# Tính errors cho từng time step
errors_by_step = {}
for model_name in all_predictions.keys():
    errors_by_step[model_name] = np.abs(y_true - all_predictions[model_name])

# Dynamic weighting: weight dựa trên inverse error tại mỗi time step
dynamic_pred = np.zeros(len(y_true))

for t in range(len(y_true)):
    # Tính weights cho time step t
    weights_t = {}
    total_inv_error = 0
    
    for model_name in all_predictions.keys():
        error_t = errors_by_step[model_name][t]
        inv_error = 1.0 / (error_t + 1e-10)
        weights_t[model_name] = inv_error
        total_inv_error += inv_error
    
    # Normalize weights
    for model_name in weights_t:
        weights_t[model_name] = weights_t[model_name] / total_inv_error
    
    # Weighted prediction tại time step t
    for model_name in all_predictions.keys():
        dynamic_pred[t] += weights_t[model_name] * all_predictions[model_name][t]

# Đánh giá
mse_dynamic = mean_squared_error(y_true, dynamic_pred)
mae_dynamic = mean_absolute_error(y_true, dynamic_pred)
rmse_dynamic = np.sqrt(mse_dynamic)
r2_dynamic = r2_score(y_true, dynamic_pred)
mape_dynamic = np.mean(np.abs((y_true - dynamic_pred) / y_true)) * 100

best_single_mse = sorted_results[0]['MSE']
improvement = ((best_single_mse - mse_dynamic) / best_single_mse) * 100

print(f"\n📊 Dynamic Weighting Results:")
print(f"   - MSE: {mse_dynamic:.4f}")
print(f"   - RMSE: {rmse_dynamic:.4f}")
print(f"   - MAE: {mae_dynamic:.4f}")
print(f"   - R²: {r2_dynamic:.4f}")
print(f"   - MAPE: {mape_dynamic:.2f}%")
print(f"   - Cải thiện so với best single: {improvement:+.2f}%")

dynamic_result = {
    'method': 'Dynamic Weighting',
    'MSE': mse_dynamic,
    'RMSE': rmse_dynamic,
    'MAE': mae_dynamic,
    'R²': r2_dynamic,
    'MAPE': mape_dynamic,
    'prediction': dynamic_pred.copy()
}


### 4.5. Residual-based Weighting


In [ ]:
print("="*70)
print("🔧 RESIDUAL-BASED WEIGHTING")
print("="*70)

# Tính residuals cho từng model
residuals = {}
for model_name in all_predictions.keys():
    residuals[model_name] = y_true - all_predictions[model_name]

# Tính các metrics về residuals
residual_metrics = {}
for model_name in all_predictions.keys():
    res = residuals[model_name]
    residual_metrics[model_name] = {
        'mean_abs_residual': np.mean(np.abs(res)),
        'std_residual': np.std(res),
        'mse_residual': np.mean(res**2),
        'bias': np.mean(res)  # Bias: positive = overestimate, negative = underestimate
    }

print("\n📊 Residual Metrics cho từng model:")
for model_name in sorted(model_names):
    metrics = residual_metrics[model_name]
    print(f"   {model_name}:")
    print(f"      - Mean |Residual|: {metrics['mean_abs_residual']:.4f}")
    print(f"      - Std Residual: {metrics['std_residual']:.4f}")
    print(f"      - MSE Residual: {metrics['mse_residual']:.4f}")
    print(f"      - Bias: {metrics['bias']:.4f}")

# Weight dựa trên inverse của mean absolute residual
weights_residual = {}
total_inv_mar = 0

for model_name in all_predictions.keys():
    mar = residual_metrics[model_name]['mean_abs_residual']
    inv_mar = 1.0 / (mar + 1e-10)
    weights_residual[model_name] = inv_mar
    total_inv_mar += inv_mar

# Normalize weights
for model_name in weights_residual:
    weights_residual[model_name] = weights_residual[model_name] / total_inv_mar

# Tạo ensemble prediction
residual_pred = np.zeros(len(y_true))
for model_name in all_predictions.keys():
    residual_pred += weights_residual[model_name] * all_predictions[model_name]

# Đánh giá
mse_residual = mean_squared_error(y_true, residual_pred)
mae_residual = mean_absolute_error(y_true, residual_pred)
rmse_residual = np.sqrt(mse_residual)
r2_residual = r2_score(y_true, residual_pred)
mape_residual = np.mean(np.abs((y_true - residual_pred) / y_true)) * 100

best_single_mse = sorted_results[0]['MSE']
improvement = ((best_single_mse - mse_residual) / best_single_mse) * 100

print(f"\n📊 Residual-based Weighting Results:")
print(f"   - MSE: {mse_residual:.4f}")
print(f"   - RMSE: {rmse_residual:.4f}")
print(f"   - MAE: {mae_residual:.4f}")
print(f"   - R²: {r2_residual:.4f}")
print(f"   - MAPE: {mape_residual:.2f}%")
print(f"   - Cải thiện so với best single: {improvement:+.2f}%")

print(f"\n📊 Weights:")
for name, weight in sorted(weights_residual.items(), key=lambda x: x[1], reverse=True):
    print(f"   - {name}: {weight:.4f}")

residual_result = {
    'method': 'Residual-based Weighting',
    'MSE': mse_residual,
    'RMSE': rmse_residual,
    'MAE': mae_residual,
    'R²': r2_residual,
    'MAPE': mape_residual,
    'prediction': residual_pred.copy()
}


## 5. So Sánh Tất Cả Phương Pháp


In [ ]:
print("="*70)
print("📊 SO SÁNH TẤT CẢ PHƯƠNG PHÁP")
print("="*70)

# Tổng hợp tất cả kết quả
all_methods = []

# Single models
for result in all_results:
    all_methods.append({
        'Method': result['Model'],
        'Type': 'Single Model',
        'MSE': result['MSE'],
        'RMSE': result['RMSE'],
        'MAE': result['MAE'],
        'R²': result['R²'],
        'MAPE': result['MAPE']
    })

# Selective Ensemble
for result in selective_ensemble_results:
    all_methods.append({
        'Method': f"Selective Top-{result['k']}",
        'Type': 'Selective Ensemble',
        'MSE': result['MSE'],
        'RMSE': result['RMSE'],
        'MAE': result['MAE'],
        'R²': result['R²'],
        'MAPE': result['MAPE']
    })

# Stacking
for result in stacking_results:
    all_methods.append({
        'Method': f"Stacking ({result['meta_learner']})",
        'Type': 'Stacking',
        'MSE': result['MSE'],
        'RMSE': result['RMSE'],
        'MAE': result['MAE'],
        'R²': result['R²'],
        'MAPE': result['MAPE']
    })

# Correlation-based
for result in correlation_ensemble_results:
    all_methods.append({
        'Method': f"Corr-based (th={result['threshold']})",
        'Type': 'Correlation Ensemble',
        'MSE': result['MSE'],
        'RMSE': result['RMSE'],
        'MAE': result['MAE'],
        'R²': result['R²'],
        'MAPE': result['MAPE']
    })

# Dynamic Weighting
all_methods.append({
    'Method': dynamic_result['method'],
    'Type': 'Dynamic Weighting',
    'MSE': dynamic_result['MSE'],
    'RMSE': dynamic_result['RMSE'],
    'MAE': dynamic_result['MAE'],
    'R²': dynamic_result['R²'],
    'MAPE': dynamic_result['MAPE']
})

# Residual-based
all_methods.append({
    'Method': residual_result['method'],
    'Type': 'Residual Weighting',
    'MSE': residual_result['MSE'],
    'RMSE': residual_result['RMSE'],
    'MAE': residual_result['MAE'],
    'R²': residual_result['R²'],
    'MAPE': residual_result['MAPE']
})

# Tạo DataFrame và sắp xếp theo MSE
comparison_df = pd.DataFrame(all_methods)
comparison_df = comparison_df.sort_values('MSE')

print("\n📈 BẢNG SO SÁNH TẤT CẢ PHƯƠNG PHÁP (sắp xếp theo MSE):")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

# Tìm best method
best_method = comparison_df.iloc[0]
best_single = comparison_df[comparison_df['Type'] == 'Single Model'].iloc[0]

print(f"\n🏆 PHƯƠNG PHÁP TỐT NHẤT: {best_method['Method']}")
print(f"   - Type: {best_method['Type']}")
print(f"   - MSE: {best_method['MSE']:.4f}")
print(f"   - RMSE: {best_method['RMSE']:.4f}")
print(f"   - MAE: {best_method['MAE']:.4f}")
print(f"   - R²: {best_method['R²']:.4f}")
print(f"   - MAPE: {best_method['MAPE']:.2f}%")

print(f"\n📊 So sánh với Best Single Model ({best_single['Method']}):")
improvement = ((best_single['MSE'] - best_method['MSE']) / best_single['MSE']) * 100
print(f"   - Best Single MSE: {best_single['MSE']:.4f}")
print(f"   - Best Method MSE: {best_method['MSE']:.4f}")
print(f"   - Cải thiện: {improvement:+.2f}%")

# Top 5 methods
print(f"\n🥇 TOP 5 PHƯƠNG PHÁP TỐT NHẤT:")
for i, row in comparison_df.head(5).iterrows():
    print(f"   {i+1}. {row['Method']} - MSE: {row['MSE']:.4f}, Type: {row['Type']}")


## 6. Visualization - So Sánh Predictions


In [ ]:
# Vẽ biểu đồ so sánh
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Plot 1: So sánh best methods vs actual
ax1 = axes[0, 0]
x_axis = np.arange(len(y_true))
ax1.plot(x_axis, y_true, 'o-', label='Thực tế', linewidth=2, markersize=4, color='black', alpha=0.8)

# Best single model
best_single_name = best_single['Method']
ax1.plot(x_axis, all_predictions[best_single_name], '-', label=f'Best Single ({best_single_name})', 
         linewidth=2, alpha=0.7, color='blue')

# Best ensemble method
if best_method['Type'] == 'Selective Ensemble':
    k = int(best_method['Method'].split('-')[1])
    best_ens_pred = next(r['prediction'] for r in selective_ensemble_results if r['k'] == k)
elif best_method['Type'] == 'Stacking':
    meta_name = best_method['Method'].split('(')[1].split(')')[0]
    best_ens_pred = next(r['prediction'] for r in stacking_results if r['meta_learner'] == meta_name)
elif best_method['Type'] == 'Correlation Ensemble':
    th = float(best_method['Method'].split('th=')[1].split(')')[0])
    best_ens_pred = next(r['prediction'] for r in correlation_ensemble_results if r['threshold'] == th)
elif best_method['Type'] == 'Dynamic Weighting':
    best_ens_pred = dynamic_result['prediction']
elif best_method['Type'] == 'Residual Weighting':
    best_ens_pred = residual_result['prediction']
else:
    best_ens_pred = None

if best_ens_pred is not None:
    ax1.plot(x_axis, best_ens_pred, '--', label=f"Best Ensemble ({best_method['Method']})", 
             linewidth=2, alpha=0.9, color='red')

ax1.set_xlabel('Time Step', fontsize=11)
ax1.set_ylabel('Giá Close', fontsize=11)
ax1.set_title('So sánh Best Methods vs Thực tế', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9, loc='best')
ax1.grid(True, alpha=0.3)

# Plot 2: MSE comparison
ax2 = axes[0, 1]
top_10 = comparison_df.head(10)
ax2.barh(range(len(top_10)), top_10['MSE'].values, color='steelblue', alpha=0.7)
ax2.set_yticks(range(len(top_10)))
ax2.set_yticklabels(top_10['Method'].values, fontsize=8)
ax2.set_xlabel('MSE', fontsize=11)
ax2.set_title('Top 10 Methods - MSE Comparison', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')
ax2.invert_yaxis()

# Plot 3: Error distribution - Best single vs Best ensemble
ax3 = axes[1, 0]
if best_ens_pred is not None:
    errors_single = y_true - all_predictions[best_single_name]
    errors_ensemble = y_true - best_ens_pred
    
    ax3.hist(errors_single, bins=30, alpha=0.6, label=f'Best Single ({best_single_name})', color='blue')
    ax3.hist(errors_ensemble, bins=30, alpha=0.6, label=f'Best Ensemble', color='red')
    ax3.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax3.set_xlabel('Error (Thực tế - Dự đoán)', fontsize=11)
    ax3.set_ylabel('Frequency', fontsize=11)
    ax3.set_title('Phân bố Error - Best Single vs Best Ensemble', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Methods by Type
ax4 = axes[1, 1]
type_summary = comparison_df.groupby('Type')['MSE'].min().sort_values()
ax4.barh(range(len(type_summary)), type_summary.values, color='green', alpha=0.7)
ax4.set_yticks(range(len(type_summary)))
ax4.set_yticklabels(type_summary.index.values, fontsize=9)
ax4.set_xlabel('Best MSE', fontsize=11)
ax4.set_title('Best MSE by Method Type', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='x')
ax4.invert_yaxis()

plt.tight_layout()
plt.show()

print("✓ Đã hiển thị biểu đồ so sánh")


## 7. Tổng Kết và Kết Luận


In [ ]:
print("="*70)
print("📊 TỔNG KẾT VÀ KẾT LUẬN")
print("="*70)

print(f"\n🎯 KẾT QUẢ NGHIÊN CỨU:")
print(f"   - Số phương pháp đã thử: {len(comparison_df)}")
print(f"   - Best Single Model: {best_single['Method']} (MSE: {best_single['MSE']:.4f})")
print(f"   - Best Overall Method: {best_method['Method']} (MSE: {best_method['MSE']:.4f})")
print(f"   - Cải thiện: {improvement:+.2f}%")

print(f"\n📈 PHÂN TÍCH:")
print(f"   - Best Single Model Type: {best_single['Type']}")
print(f"   - Best Ensemble Type: {best_method['Type']}")

# Phân tích theo type
print(f"\n📊 BEST METHOD THEO TỪNG TYPE:")
for method_type in comparison_df['Type'].unique():
    type_df = comparison_df[comparison_df['Type'] == method_type]
    best_in_type = type_df.iloc[0]
    print(f"   - {method_type}: {best_in_type['Method']} (MSE: {best_in_type['MSE']:.4f})")

# Kết luận
print(f"\n💡 KẾT LUẬN:")
if improvement > 0:
    print(f"   ✓ Ensemble method cải thiện được {improvement:.2f}% so với best single model")
    print(f"   ✓ Phương pháp tốt nhất: {best_method['Method']}")
    print(f"   ✓ Nên sử dụng: {best_method['Method']} cho submission")
else:
    print(f"   ⚠️  Ensemble method không cải thiện so với best single model")
    print(f"   ⚠️  Nên sử dụng: {best_single['Method']} cho submission")
    print(f"   💡 Có thể do:")
    print(f"      - Các models có correlation cao")
    print(f"      - Best single model đã rất tốt")
    print(f"      - Cần tối ưu hyperparameters tốt hơn")

print(f"\n✓ Hoàn thành nghiên cứu ensemble methods!")
